# Movie 6 bis — Ensemble SVM + Twitter-RoBERTa (config RoBERTa ancienne version)

Ce notebook reprend :

- **ta configuration RoBERTa qui marchait**
- **le meilleur SVM classique**
- une **fusion simple** : suivre RoBERTa si sa confiance est assez haute, sinon suivre SVM
- une **recherche du meilleur seuil** sur validation
- un **réentraînement complet** sur tout le train
- la génération de **3 fichiers de soumission** :
  - SVM
  - Twitter-RoBERTa
  - Ensemble

L'idée est de repartir d'une base stable et propre, sans les bugs rencontrés dans le notebook précédent.


In [2]:
# Si besoin sur Colab, décommente :
!pip install -q transformers datasets accelerate scikit-learn

from pathlib import Path
import os
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed(42)

## Configuration

In [4]:
# ===== Chemins =====
DATA_DIR = Path("/content/drive/MyDrive/projet tal/movies1000/movies1000")
TEST_FILE = Path("/content/drive/MyDrive/projet tal/test.txt")
OUTPUT_DIR = Path("/content/drive/MyDrive/projet tal/movie6bis_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ===== Labels =====
label2id = {"N": 0, "P": 1}
id2label = {0: "N", 1: "P"}

# ===== Modèle RoBERTa (ancienne config) =====
ROBERTA_MODEL_NAME = "cardiffnlp/twitter-roberta-base-sentiment-latest"
ROBERTA_RUN_NAME = "twitter_roberta_cardiffnlp_ensemble_oldconfig"
ROBERTA_NUM_EPOCHS = 2
ROBERTA_LEARNING_RATE = 2e-5
ROBERTA_BATCH_TRAIN = 8
ROBERTA_BATCH_EVAL = 16
ROBERTA_WEIGHT_DECAY = 0.01
ROBERTA_MAX_LENGTH = 512

# ===== Meilleur SVM classique =====
SVM_NGRAM_RANGE = (1, 2)
SVM_MIN_DF = 3
SVM_MAX_DF = 0.95
SVM_SUBLINEAR_TF = True
SVM_C = 2.0

# ===== Ensemble =====
THRESHOLD_GRID = np.round(np.arange(0.50, 1.001, 0.01), 2)

# ===== Divers =====
MAKE_TEST_SUBMISSIONS = True
SAVE_TRAINED_MODELS = True

In [5]:
print("DATA_DIR existe :", DATA_DIR.exists())
if DATA_DIR.exists():
    print("Contenu :", [p.name for p in DATA_DIR.iterdir()])

print("TEST_FILE existe :", TEST_FILE.exists())
print("OUTPUT_DIR :", OUTPUT_DIR)

DATA_DIR existe : True
Contenu : ['neg', 'pos']
TEST_FILE existe : False
OUTPUT_DIR : /content/drive/MyDrive/projet tal/movie6bis_outputs


## Chargement des données

In [6]:
def load_movies_from_folder(data_dir: Path) -> pd.DataFrame:
    rows = []

    for label_folder, label in [("pos", "P"), ("neg", "N")]:
        folder = data_dir / label_folder
        if not folder.exists():
            continue

        for file_path in folder.glob("*.txt"):
            text = file_path.read_text(encoding="utf-8", errors="ignore")
            rows.append({
                "doc_id": file_path.name,
                "label": label,
                "text": text,
            })

    if not rows:
        raise ValueError("Aucun fichier trouvé. Vérifie DATA_DIR.")

    return pd.DataFrame(rows).sort_values("doc_id").reset_index(drop=True)

def load_test_reviews(test_file: Path) -> pd.DataFrame:
    if not test_file.exists():
        print("Fichier test introuvable :", test_file)
        return None

    lines = test_file.read_text(encoding="utf-8", errors="ignore").splitlines()
    lines = [line.strip() for line in lines if line.strip() != ""]
    return pd.DataFrame({
        "row_id": np.arange(len(lines)),
        "text": lines,
    })

df = load_movies_from_folder(DATA_DIR)
test_df = load_test_reviews(TEST_FILE)

print("Train complet :", df.shape)
display(df.head())

if test_df is not None:
    print("Test :", test_df.shape)
    display(test_df.head())

Fichier test introuvable : /content/drive/MyDrive/projet tal/test.txt
Train complet : (2000, 3)


,doc_id,label,text
0,cv000_29416.txt,N,"plot : two teen couples go to a church party ,..."
1,cv000_29590.txt,P,films adapted from comic books have had plenty...
2,cv001_18431.txt,P,every now and then a movie comes along from a ...
3,cv001_19502.txt,N,the happy bastard's quick movie review \ndamn ...
4,cv002_15918.txt,P,you've got mail works alot better than it dese...


## Split validation

In [7]:
train_df, valid_df = train_test_split(
    df[["doc_id", "label", "text"]].copy(),
    test_size=0.2,
    random_state=42,
    stratify=df["label"],
)

train_df = train_df.reset_index(drop=True)
valid_df = valid_df.reset_index(drop=True)

print("Train :", train_df.shape)
print("Valid :", valid_df.shape)
print(train_df["label"].value_counts().sort_index())

Train : (1600, 3)
Valid : (400, 3)
label
N    800
P    800
Name: count, dtype: int64


## Jeux Hugging Face pour RoBERTa

In [8]:
hf_train_df = train_df[["text", "label"]].copy()
hf_valid_df = valid_df[["text", "label"]].copy()

hf_train_df["label_id"] = hf_train_df["label"].map(label2id)
hf_valid_df["label_id"] = hf_valid_df["label"].map(label2id)

hf_train = Dataset.from_pandas(
    hf_train_df[["text", "label_id"]].rename(columns={"label_id": "label"})
)
hf_valid = Dataset.from_pandas(
    hf_valid_df[["text", "label_id"]].rename(columns={"label_id": "label"})
)

hf_train, hf_valid

(Dataset({
     features: ['text', 'label'],
     num_rows: 1600
 }),
 Dataset({
     features: ['text', 'label'],
     num_rows: 400
 }))

## Métriques et utilitaires

In [9]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    y_true = np.array([id2label[int(x)] for x in labels])
    y_pred = np.array([id2label[int(x)] for x in preds])

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, pos_label="P"),
        "recall": recall_score(y_true, y_pred, pos_label="P"),
        "f1": f1_score(y_true, y_pred, pos_label="P"),
    }

def print_report(y_true, y_pred, title):
    print(f"\n===== {title} =====")
    print(classification_report(y_true, y_pred, digits=4))
    metrics = pd.DataFrame({
        "score": {
            "accuracy": accuracy_score(y_true, y_pred),
            "precision": precision_score(y_true, y_pred, pos_label="P"),
            "recall": recall_score(y_true, y_pred, pos_label="P"),
            "f1": f1_score(y_true, y_pred, pos_label="P"),
        }
    })
    display(metrics)

def softmax_np(x):
    x = np.asarray(x)
    x = x - np.max(x, axis=1, keepdims=True)
    exp_x = np.exp(x)
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

## Entraînement SVM

In [10]:
def train_svm(train_texts, train_labels):
    model = Pipeline([
        ("tfidf", TfidfVectorizer(
            ngram_range=SVM_NGRAM_RANGE,
            min_df=SVM_MIN_DF,
            max_df=SVM_MAX_DF,
            sublinear_tf=SVM_SUBLINEAR_TF,
        )),
        ("clf", LinearSVC(C=SVM_C))
    ])
    model.fit(train_texts, train_labels)
    return model

svm_model = train_svm(train_df["text"], train_df["label"])
valid_pred_svm = svm_model.predict(valid_df["text"])

print_report(valid_df["label"], valid_pred_svm, "LinearSVC (validation)")


===== LinearSVC (validation) =====
              precision    recall  f1-score   support

           N     0.9062    0.8700    0.8878       200
           P     0.8750    0.9100    0.8922       200

    accuracy                         0.8900       400
   macro avg     0.8906    0.8900    0.8900       400
weighted avg     0.8906    0.8900    0.8900       400



,score
accuracy,0.890000
precision,0.875000
recall,0.910000
f1,0.892157


## Entraînement RoBERTa — ancienne configuration

In [11]:
def run_transformer_experiment(
    model_name: str,
    run_name: str,
    hf_train,
    hf_valid,
    valid_df,
    num_epochs: int = 2,
    learning_rate: float = 2e-5,
    batch_size_train: int = 8,
    batch_size_eval: int = 16,
    max_length: int = 512,
):
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    def tokenize_fn(batch):
        return tokenizer(
            batch["text"],
            truncation=True,
            max_length=max_length,
        )

    tokenized_train = hf_train.map(tokenize_fn, batched=True)
    tokenized_valid = hf_valid.map(tokenize_fn, batched=True)

    if "text" in tokenized_train.column_names:
        tokenized_train = tokenized_train.remove_columns(["text"])
    if "text" in tokenized_valid.column_names:
        tokenized_valid = tokenized_valid.remove_columns(["text"])

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2,
        id2label=id2label,
        label2id=label2id,
        ignore_mismatched_sizes=True,
    )

    output_dir = str(OUTPUT_DIR / f"{run_name}_output")

    training_args = TrainingArguments(
        output_dir=output_dir,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="epoch",
        learning_rate=learning_rate,
        per_device_train_batch_size=batch_size_train,
        per_device_eval_batch_size=batch_size_eval,
        num_train_epochs=num_epochs,
        weight_decay=ROBERTA_WEIGHT_DECAY,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        report_to="none",
        save_total_limit=1,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_valid,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    train_output = trainer.train()
    eval_output = trainer.evaluate()

    pred_output = trainer.predict(tokenized_valid)
    logits = pred_output.predictions
    probs = softmax_np(logits)
    pred_ids = np.argmax(logits, axis=1)
    pred_labels = np.array([id2label[int(x)] for x in pred_ids])

    print_report(valid_df["label"], pred_labels, run_name)

    return {
        "tokenizer": tokenizer,
        "trainer": trainer,
        "train_output": train_output,
        "eval_output": eval_output,
        "pred_labels": pred_labels,
        "pred_probs": probs,
        "tokenized_valid": tokenized_valid,
    }

twitter_roberta_results = run_transformer_experiment(
    model_name=ROBERTA_MODEL_NAME,
    run_name=ROBERTA_RUN_NAME,
    hf_train=hf_train,
    hf_valid=hf_valid,
    valid_df=valid_df,
    num_epochs=ROBERTA_NUM_EPOCHS,
    learning_rate=ROBERTA_LEARNING_RATE,
    batch_size_train=ROBERTA_BATCH_TRAIN,
    batch_size_eval=ROBERTA_BATCH_EVAL,
    max_length=ROBERTA_MAX_LENGTH,
)

config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Map:   0%|          | 0/1600 [00:00<?, ? examples/s]

...

pytorch_model.bin:   0%|          | 0.00/501M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/501M [00:00<?, ?B/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |                                                                                     
--------------------------------+------------+-------------------------------------------------------------------------------------
roberta.pooler.dense.weight     | UNEXPECTED |                                                                                     
roberta.pooler.dense.bias       | UNEXPECTED |                                                                                     
roberta.embeddings.position_ids | UNEXPECTED |                                                                                     
classifier.out_proj.weight      | MISMATCH   | Reinit due to size mismatch ckpt: torch.Size([3, 768]) vs model:torch.Size([2, 768])
classifier.out_proj.bias        | MISMATCH   | Reinit due to size mismatch ckpt: torch.Size([3]) vs model:to

Epoch,Training Loss,Validation Loss


## Comparaison validation : SVM vs RoBERTa

In [ ]:
valid_pred_roberta = twitter_roberta_results["pred_labels"]
valid_probs_roberta = twitter_roberta_results["pred_probs"]
valid_roberta_conf = valid_probs_roberta.max(axis=1)

comparison_df = pd.DataFrame({
    "y_true": valid_df["label"].values,
    "svm_pred": valid_pred_svm,
    "roberta_pred": valid_pred_roberta,
    "roberta_conf": valid_roberta_conf,
})

comparison_df["svm_correct"] = comparison_df["svm_pred"] == comparison_df["y_true"]
comparison_df["roberta_correct"] = comparison_df["roberta_pred"] == comparison_df["y_true"]
comparison_df["disagree"] = comparison_df["svm_pred"] != comparison_df["roberta_pred"]

display(comparison_df.head())

summary = pd.DataFrame({
    "metric": ["svm_acc", "svm_f1_P", "roberta_acc", "roberta_f1_P", "disagreement_rate"],
    "value": [
        accuracy_score(comparison_df["y_true"], comparison_df["svm_pred"]),
        f1_score(comparison_df["y_true"], comparison_df["svm_pred"], pos_label="P"),
        accuracy_score(comparison_df["y_true"], comparison_df["roberta_pred"]),
        f1_score(comparison_df["y_true"], comparison_df["roberta_pred"], pos_label="P"),
        comparison_df["disagree"].mean(),
    ]
})
display(summary)

## Recherche du meilleur seuil pour l'ensemble

In [ ]:
threshold_rows = []

for thr in THRESHOLD_GRID:
    ens_pred = np.where(
        valid_roberta_conf >= thr,
        valid_pred_roberta,
        valid_pred_svm,
    )
    threshold_rows.append({
        "threshold": thr,
        "accuracy": accuracy_score(valid_df["label"], ens_pred),
        "precision": precision_score(valid_df["label"], ens_pred, pos_label="P"),
        "recall": recall_score(valid_df["label"], ens_pred, pos_label="P"),
        "f1": f1_score(valid_df["label"], ens_pred, pos_label="P"),
    })

threshold_df = pd.DataFrame(threshold_rows).sort_values(
    ["f1", "accuracy", "threshold"], ascending=[False, False, False]
).reset_index(drop=True)

display(threshold_df.head(10))

BEST_THRESHOLD = float(threshold_df.loc[0, "threshold"])
print("Meilleur seuil validation =", BEST_THRESHOLD)

valid_pred_ensemble = np.where(
    valid_roberta_conf >= BEST_THRESHOLD,
    valid_pred_roberta,
    valid_pred_svm,
)

print_report(valid_df["label"], valid_pred_ensemble, f"Ensemble validation (threshold={BEST_THRESHOLD})")

## Erreurs sur les désaccords

In [ ]:
disagree_df = comparison_df[comparison_df["disagree"]].copy()
print("Nombre de désaccords :", len(disagree_df))

if len(disagree_df) > 0:
    print("Accuracy SVM sur désaccords :", accuracy_score(disagree_df["y_true"], disagree_df["svm_pred"]))
    print("Accuracy RoBERTa sur désaccords :", accuracy_score(disagree_df["y_true"], disagree_df["roberta_pred"]))
    print("Confiance moyenne RoBERTa sur désaccords :", disagree_df["roberta_conf"].mean())

display(disagree_df.head(20))

## Sauvegarde optionnelle du modèle RoBERTa validation

In [ ]:
if SAVE_TRAINED_MODELS:
    save_dir = OUTPUT_DIR / "best_twitter_roberta_valid_model"
    twitter_roberta_results["trainer"].save_model(str(save_dir))
    twitter_roberta_results["tokenizer"].save_pretrained(str(save_dir))
    print("Modèle et tokenizer sauvegardés dans :", save_dir)

## Réentraînement complet pour les soumissions

In [ ]:
def train_full_svm(df_full: pd.DataFrame):
    model = Pipeline([
        ("tfidf", TfidfVectorizer(
            ngram_range=SVM_NGRAM_RANGE,
            min_df=SVM_MIN_DF,
            max_df=SVM_MAX_DF,
            sublinear_tf=SVM_SUBLINEAR_TF,
        )),
        ("clf", LinearSVC(C=SVM_C))
    ])
    model.fit(df_full["text"], df_full["label"])
    return model

def train_full_roberta(df_full: pd.DataFrame, tokenizer):
    df_roberta = df_full[["text", "label"]].copy()
    df_roberta["label_id"] = df_roberta["label"].map(label2id).astype(int)

    hf_full = Dataset.from_pandas(
        df_roberta[["text", "label_id"]].rename(columns={"label_id": "label"})
    )

    def tokenize_full(batch):
        return tokenizer(
            batch["text"],
            truncation=True,
            max_length=ROBERTA_MAX_LENGTH,
        )

    tokenized_full = hf_full.map(tokenize_full, batched=True)

    cols_to_remove = [c for c in tokenized_full.column_names if c in ["text", "__index_level_0__"]]
    if cols_to_remove:
        tokenized_full = tokenized_full.remove_columns(cols_to_remove)

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    model = AutoModelForSequenceClassification.from_pretrained(
        ROBERTA_MODEL_NAME,
        num_labels=2,
        id2label=id2label,
        label2id=label2id,
        ignore_mismatched_sizes=True,
    )

    training_args = TrainingArguments(
        output_dir=str(OUTPUT_DIR / f"{ROBERTA_RUN_NAME}_full_output"),
        eval_strategy="no",
        save_strategy="no",
        logging_strategy="epoch",
        learning_rate=ROBERTA_LEARNING_RATE,
        per_device_train_batch_size=ROBERTA_BATCH_TRAIN,
        per_device_eval_batch_size=ROBERTA_BATCH_EVAL,
        num_train_epochs=ROBERTA_NUM_EPOCHS,
        weight_decay=ROBERTA_WEIGHT_DECAY,
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_full,
        data_collator=data_collator,
    )

    trainer.train()
    return trainer

def predict_roberta_on_test(trainer, tokenizer, test_df):
    hf_test = Dataset.from_pandas(test_df[["text"]].copy())

    def tokenize_test(batch):
        return tokenizer(
            batch["text"],
            truncation=True,
            max_length=ROBERTA_MAX_LENGTH,
        )

    tokenized_test = hf_test.map(tokenize_test, batched=True)
    cols_to_remove = [c for c in tokenized_test.column_names if c in ["text", "__index_level_0__"]]
    if cols_to_remove:
        tokenized_test = tokenized_test.remove_columns(cols_to_remove)

    output = trainer.predict(tokenized_test)
    logits = output.predictions
    probs = softmax_np(logits)
    pred_ids = np.argmax(logits, axis=1)
    pred_labels = np.array([id2label[int(x)] for x in pred_ids])

    return pred_labels, probs

def save_submission(labels, path: Path):
    sub = pd.DataFrame({"label": labels})
    sub.to_csv(path, index=False)
    print("Soumission enregistrée :", path)
    print(sub["label"].value_counts())
    return sub

## Génération des soumissions test

In [ ]:
if MAKE_TEST_SUBMISSIONS and (test_df is not None):
    # 1) Full SVM
    full_svm = train_full_svm(df)
    test_pred_svm = full_svm.predict(test_df["text"])

    # 2) Full RoBERTa avec la même config que l'ancienne version
    full_tokenizer = AutoTokenizer.from_pretrained(ROBERTA_MODEL_NAME)
    full_roberta_trainer = train_full_roberta(df, full_tokenizer)
    test_pred_roberta, test_probs_roberta = predict_roberta_on_test(
        full_roberta_trainer,
        full_tokenizer,
        test_df
    )

    # 3) Ensemble avec meilleur seuil trouvé sur validation
    test_roberta_conf = test_probs_roberta.max(axis=1)
    test_pred_ensemble = np.where(
        test_roberta_conf >= BEST_THRESHOLD,
        test_pred_roberta,
        test_pred_svm,
    )

    sub_svm = save_submission(test_pred_svm, OUTPUT_DIR / "submission_movie6bis_svm.csv")
    sub_roberta = save_submission(test_pred_roberta, OUTPUT_DIR / "submission_movie6bis_twitter_roberta.csv")
    sub_ensemble = save_submission(test_pred_ensemble, OUTPUT_DIR / "submission_movie6bis_ensemble.csv")

    display(sub_svm.head())
    display(sub_roberta.head())
    display(sub_ensemble.head())
else:
    print("Soumissions non générées : vérifie MAKE_TEST_SUBMISSIONS et TEST_FILE.")

## Lecture rapide des résultats

Après exécution, regarde dans cet ordre :

1. **Validation SVM**
2. **Validation RoBERTa**
3. **Tableau des seuils**
4. **Performance de l'ensemble**
5. **Fichiers de soumission générés**

### Conseils
- Si **RoBERTa seul** > ensemble > SVM : soumets d'abord RoBERTa, puis ensemble si tu as une autre tentative.
- Si **SVM** > ensemble > RoBERTa : soumets d'abord SVM.
- Si **ensemble** est meilleur en validation et cohérent, essaie-le sur la plateforme.
